# Walker2d PPO：从倒立摆迁移到双足控制

这本 Notebook 是项目的顺序学习入口。每节采用“目标 → 关键解释 → 可运行代码 → 观察问题”。核心算法保留在 `src/walker2d_ppo/`，便于测试；Notebook 会直接调用并检查这些实现。

**边界声明**：`smoke` 只验证训练闭环，不代表策略学会行走；Stage C/D 的结论必须来自真实实验数据。

## 00｜依赖、随机种子与设备

首版固定 CPU。MuJoCo 物理仿真主要在 CPU，小型网络放到 MPS 不一定更快。

In [ ]:
import json, platform, sys, time
from pathlib import Path

import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch

from walker2d_ppo.algorithm import compute_gae
from walker2d_ppo.config import profile
from walker2d_ppo.evaluate import evaluate_policy
from walker2d_ppo.model import GaussianActor, ValueCritic
from walker2d_ppo.rollout import RolloutCollector
from walker2d_ppo.trainer import load_actor, train
from walker2d_ppo.update import ppo_update

SEED = 1
DEVICE = torch.device('cpu')
np.random.seed(SEED)
torch.manual_seed(SEED)
print({
    'python': sys.version.split()[0], 'platform': platform.platform(),
    'gymnasium': gym.__version__, 'torch': torch.__version__, 'device': str(DEVICE),
})

## 01｜Walker2d 环境、17 维观测与 6 维动作

默认观测是 8 个位置量（不含绝对 x）+ 9 个速度量。动作是六个铰链的力矩：右髋/膝/踝、左髋/膝/踝；它不是目标关节角度 + PD。

In [ ]:
OBSERVATIONS = [
    'torso height z', 'torso angle',
    'right thigh angle', 'right leg angle', 'right foot angle',
    'left thigh angle', 'left leg angle', 'left foot angle',
    'torso x velocity', 'torso z velocity', 'torso angular velocity',
    'right thigh angular velocity', 'right leg angular velocity', 'right foot angular velocity',
    'left thigh angular velocity', 'left leg angular velocity', 'left foot angular velocity',
]
ACTIONS = [
    'right thigh torque', 'right leg torque', 'right foot torque',
    'left thigh torque', 'left leg torque', 'left foot torque',
]
env = gym.make('Walker2d-v5')
obs, reset_info = env.reset(seed=SEED)
next_obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print('observation_space:', env.observation_space)
print('action_space:', env.action_space)
print('reset obs:', obs.shape, obs.dtype)
print('network input dtype:', np.asarray(obs, dtype=np.float32).dtype)
print('step:', next_obs.shape, reward, terminated, truncated)
print('reward terms:', {k: info[k] for k in ('reward_forward','reward_ctrl','reward_survive')})
print('dt:', env.unwrapped.dt, '= timestep', env.unwrapped.model.opt.timestep,
      '* frame_skip', env.unwrapped.frame_skip)
assert obs.shape == (17,) and env.action_space.shape == (6,)
assert np.isclose(reward, info['reward_forward'] + info['reward_ctrl'] + info['reward_survive'])
for i, name in enumerate(OBSERVATIONS): print(f'{i:2d}: {name:34s} {obs[i]: .5f}')
env.close()

**观察问题**

1. 哪些量可由编码器/IMU 直接或间接估计？哪些是仿真精确状态？  
2. 为什么删除绝对 x 位置有利于学习与位置无关的策略？  
3. `reward_ctrl` 已是负数，为什么不能再减一次？

## 02｜随机策略、摔倒与可视化

随机动作通常很快使躯干高度或角度越界。下面保存一个短 GIF；若本机渲染不可用，训练仍可继续。

In [ ]:
video_path = Path('outputs/random_policy.gif')
video_path.parent.mkdir(exist_ok=True)
render_env = gym.make('Walker2d-v5', render_mode='rgb_array')
obs, info = render_env.reset(seed=SEED)
frames, heights, angles, x_velocities = [], [], [], []
episode_return = 0.0
for step in range(200):
    obs, reward, terminated, truncated, info = render_env.step(render_env.action_space.sample())
    frame = render_env.render()
    if frame is not None: frames.append(frame)
    heights.append(float(obs[0])); angles.append(float(obs[1]))
    x_velocities.append(float(obs[8])); episode_return += float(reward)
    if terminated or truncated: break
render_env.close()
if frames:
    imageio.mimsave(video_path, frames, fps=30, loop=0)
print({'return': episode_return, 'length': len(heights), 'terminated': terminated,
       'truncated': truncated, 'video': str(video_path) if frames else 'render unavailable'})
plt.plot(heights, label='torso height')
plt.plot(angles, label='torso angle')
plt.axhline(0.8, color='gray', linestyle='--', linewidth=1)
plt.legend(); plt.xlabel('environment step'); plt.title('Random policy: fall diagnostics'); plt.show()

## 03｜Actor / Critic 与联合 log probability

PPO 的概率比必须使用六维 raw action 的**联合** log probability，即 `dist.log_prob(action).sum(-1)`。

In [ ]:
actor = GaussianActor(obs_dim=17, action_dim=6).to(DEVICE)
critic = ValueCritic(obs_dim=17).to(DEVICE)
batch_obs = torch.zeros(4, 17, device=DEVICE)
raw_action, old_log_prob, entropy = actor.sample(batch_obs)
new_log_prob = actor.joint_log_prob(actor.distribution(batch_obs), raw_action)
ratio = (new_log_prob - old_log_prob).exp()
print('raw_action', raw_action.shape, 'joint log_prob', old_log_prob.shape,
      'value', critic(batch_obs).shape)
print('ratio before update:', ratio)
assert raw_action.shape == (4, 6) and old_log_prob.shape == (4,)
torch.testing.assert_close(ratio, torch.ones_like(ratio))

**动作一致性**

Buffer 保存 raw action `u` 与 `log π_old(u|s)`；环境执行 `clip(u,-1,1)`；更新时仍评价同一个 `u`。这不是裁剪后动作的精确密度，但避免了“存 clip 后动作、却用原高斯密度”这一不一致错误。

## 04｜Rollout Buffer 与跨 rollout Episode

collector 是有状态对象。收集长度到了只切开训练 batch，不重置环境，也不清零 Episode Return/Length。

In [ ]:
env = gym.make('Walker2d-v5')
collector = RolloutCollector(env, actor, critic, DEVICE, seed=SEED)
rollout = collector.collect(steps=128, gamma=0.99, gae_lambda=0.95)
rollout.assert_valid(obs_dim=17, action_dim=6)
print({
    'obs': tuple(rollout.observations.shape), 'raw_action': tuple(rollout.raw_actions.shape),
    'log_prob': tuple(rollout.old_log_probs.shape), 'completed_episodes': len(rollout.episodes),
    'saturation': float((rollout.env_actions.abs() >= .999).float().mean()),
})

## 05｜TD / GAE：terminated、truncated 与 rollout 边界

`terminated` 决定 TD 是否 bootstrap；`terminated or truncated` 决定是否允许 GAE 接到下一个样本。最后一个 rollout 样本总会停止数组递推，但仍用它自己的 `next_value`。

In [ ]:
# t=0 模拟 TimeLimit：bootstrap V(next)=4，但不把 t=1 的优势接回来。
adv, ret = compute_gae(
    rewards=torch.tensor([1., 2.]), values=torch.tensor([.5, .25]),
    next_values=torch.tensor([4., 7.]),
    terminated=torch.tensor([False, False]),
    episode_ends=torch.tensor([True, False]),
    gamma=.9, gae_lambda=1.,
)
print('advantages:', adv, 'returns:', ret)
torch.testing.assert_close(adv[0], torch.tensor(4.1))
torch.testing.assert_close(ret[0], torch.tensor(4.6))
assert not adv.requires_grad and not ret.requires_grad

## 06｜一次 PPO 更新

旧数据全部 detach；只有当前 Actor/Critic 的预测进入梯度。指标是本 rollout 内多个 epoch/mini-batch 的聚合值。

In [ ]:
config = profile('smoke')
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=config.actor_lr)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=config.critic_lr)
metrics = ppo_update(actor, critic, actor_optimizer, critic_optimizer, rollout, config)
print(metrics)
assert all(np.isfinite(v) for v in [metrics.actor_loss, metrics.critic_loss,
                                     metrics.approx_kl, metrics.entropy])
env.close()

## 07｜基线训练与日志仪表盘

默认执行 128 步 smoke 并保存 checkpoint。想观察趋势时，把档位改成 `short`；不要把 smoke 的评估当作收敛结果。

In [ ]:
RUN_PROFILE = 'smoke'  # 可改为 'short'；full 前请先评估本机吞吐
run_dir = Path(f'runs/notebook-{RUN_PROFILE}')
checkpoint_path = train(profile(RUN_PROFILE, output_dir=run_dir))
print('checkpoint:', checkpoint_path)

metrics_data = np.genfromtxt(run_dir / 'metrics.csv', delimiter=',', names=True)
metrics_data = np.atleast_1d(metrics_data)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(metrics_data['env_steps'], metrics_data['mean_rollout_reward'])
axes[0].set_title('Mean step reward')
axes[1].plot(metrics_data['env_steps'], metrics_data['actor_loss'], label='actor')
axes[1].plot(metrics_data['env_steps'], metrics_data['critic_loss'], label='critic')
axes[1].legend(); axes[1].set_title('Update losses')
axes[2].plot(metrics_data['env_steps'], metrics_data['action_saturation'])
axes[2].set_title('Executed action saturation')
for ax in axes: ax.set_xlabel('environment steps')
plt.tight_layout(); plt.show()

## 08｜固定 seed 独立评估与回放

使用 Actor 均值作为确定性动作，不采样、不更新参数。Return、Length、距离和控制能量分开报告。

In [ ]:
trained_actor, saved = load_actor(checkpoint_path, DEVICE)
eval_gif = Path(f'outputs/{RUN_PROFILE}-evaluation.gif')
evaluation = evaluate_policy(
    trained_actor, 'Walker2d-v5', seeds=[10000, 10001, 10002], device=DEVICE,
    env_kwargs={
        'forward_reward_weight': saved['config']['forward_reward_weight'],
        'ctrl_cost_weight': saved['config']['ctrl_cost_weight'],
        'healthy_reward': saved['config']['healthy_reward'],
    },
    video_path=eval_gif,
)
print(json.dumps(evaluation.summary(), indent=2))
print('video:', eval_gif)

## 09｜奖励设计对照实验

先写假设，再一次只改一个权重。不同奖励定义下的训练 Return 不能直接比较；统一比较固定评估条件下的 Length、前进距离、控制能量与视频。

推荐 C1：`ctrl_cost_weight ∈ {1e-4, 1e-3, 1e-2}`。控制惩罚增大可能降低动作能量，也可能削弱保持平衡的能力。详见 `EXPERIMENT_GUIDE.md`。

In [ ]:
experiment_configs = {
    'low_ctrl_cost': profile('short', ctrl_cost_weight=1e-4, output_dir=Path('runs/c1/low')),
    'baseline': profile('short', ctrl_cost_weight=1e-3, output_dir=Path('runs/c1/base')),
    'high_ctrl_cost': profile('short', ctrl_cost_weight=1e-2, output_dir=Path('runs/c1/high')),
}
for name, cfg in experiment_configs.items():
    print(name, cfg.ctrl_cost_weight, cfg.total_timesteps, cfg.seed)
# 明确取消下一行注释后才会依次训练：
# checkpoints = {name: train(cfg) for name, cfg in experiment_configs.items()}

## 10｜鲁棒性扩展（纯仿真，不等于 Sim-to-Real）

在策略固定后，用标称、较低、较高地面摩擦或连杆质量做测试时扰动。先记录训练/测试分布，再比较距离、Length 和能量。真实部署还要面对噪声、延迟、执行器带宽和未建模接触，因此这里不能声称完成真实硬件迁移。

### 最终自检

- 能否用自己的话解释 17 维观测和 6 维力矩？
- 能否画出 raw action → clip → environment 与 log probability 的数据流？
- 能否说明 `terminated` 与 `truncated` 的两个 mask？
- 能否从视频和日志区分代码错误、训练不足与奖励设计问题？
- 能否提出一个只改变一个因素、使用共同外部指标的实验？